# Baseline Models - House Prices (Linear Regression, Decision Tree)


- Dữ liệu: `Ames/data/processed` ( `random_state=42`)
- Target: `log1p(SalePrice)`  (RMSLE)
- Tập valid = `X_test.csv` / `y_test.csv` (, tập test Kaggle = `X_kaggle_test.csv`

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 1. Đọc dữ liệu

In [2]:
DATA = Path('../../data/processed')
OUT = Path('../../../outputdata')
OUT.mkdir(exist_ok=True)

X_train = pd.read_csv(DATA / 'X_train.csv')
X_valid = pd.read_csv(DATA / 'X_test.csv')
X_kaggle = pd.read_csv(DATA / 'X_kaggle_test.csv')
y_train = pd.read_csv(DATA / 'y_train.csv')['SalePrice']
y_valid = pd.read_csv(DATA / 'y_test.csv')['SalePrice']
ids = pd.read_csv('../../data/raw/sample_submission.csv')['Id']

print(X_train.shape, X_valid.shape, X_kaggle.shape)
assert (X_train.columns == X_kaggle.columns).all()
assert len(ids) == len(X_kaggle)

(1166, 146) (292, 146) (1459, 146)


## 2. Train và đánh giá trên train, valid

Dự đoán trên thang log rồi đổi về giá thật bằng `expm1` để tính RMSE, MAE, R². RMSLE là chỉ số Kaggle dùng.

In [3]:
def evaluate(y_true, y_pred):
    return {
        'RMSLE': np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
}

rows = []
kaggle_pred = {}
for name, model in models.items():
    model.fit(X_train, np.log1p(y_train))
    for split, X, y in [('train', X_train, y_train), ('valid', X_valid, y_valid)]:
        pred = np.expm1(model.predict(X))
        rows.append({'model': name, 'split': split, **evaluate(y, pred)})
    kaggle_pred[name] = np.expm1(model.predict(X_kaggle))

scores = pd.DataFrame(rows).round(5)
scores

,model,split,RMSLE,RMSE,MAE,R2
0,Linear Regression,train,0.11534,21632.12072,14772.25487,0.92815
1,Linear Regression,valid,0.15445,27084.97112,18905.89213,0.86719
2,Decision Tree,train,0.00166,231.67407,13.55060,0.99999
3,Decision Tree,valid,0.22693,43622.46789,29929.12329,0.65550


## 3. Tạo file nộp Kaggle

In [4]:
files = {'Linear Regression': 'submission_linear_regression.csv', 'Decision Tree': 'submission_decision_tree.csv'}
for name, fname in files.items():
    sub = pd.DataFrame({'Id': ids, 'SalePrice': kaggle_pred[name]})
    sub.to_csv(OUT / fname, index=False)
    print(fname, sub.shape, 'min', round(sub.SalePrice.min()), 'max', round(sub.SalePrice.max()), 'NaN', sub.SalePrice.isna().sum())

submission_linear_regression.csv (1459, 2) min 38770 max 1373249 NaN 0
submission_decision_tree.csv (1459, 2) min 34900 max 582933 NaN 0


## 4. Bảng so sánh valid và Kaggle



In [5]:
kaggle_score = {'Linear Regression': 0.16710, 'Decision Tree': 0.22470}

valid = scores[scores.split == 'valid'].set_index('model')['RMSLE']
train = scores[scores.split == 'train'].set_index('model')['RMSLE']
summary = pd.DataFrame({
    'RMSLE train': train,
    'RMSLE valid': valid,
    'Kaggle public score': pd.Series(kaggle_score),
}).round(5)
summary.to_csv('baseline_summary.csv')
scores.to_csv('baseline_scores.csv', index=False)
summary

,RMSLE train,RMSLE valid,Kaggle public score
Linear Regression,0.11534,0.15445,0.1671
Decision Tree,0.00166,0.22693,0.2247


## 5. Nhận xét

- Linear Regression tốt hơn Decision Tree ở cả valid (0.1545 so với 0.2272) và Kaggle (0.1671 so với 0.2247).
- Decision Tree không giới hạn độ sâu nên học thuộc tập train (RMSLE train gần 0), valid và Kaggle đều cao: overfitting rõ rệt.
- Điểm Kaggle sát điểm valid (chênh khoảng 0.013 với Linear, 0.003 với Decision Tree), nên tập valid tự chia phản ánh khá đúng điểm thật.
- Linear Regression có một dự đoán rất lớn trên tập test Kaggle (khoảng 1.37 triệu USD, trong khi giá cao nhất ở train là 755k), có thể làm điểm Kaggle xấu hơn valid.

## 6. So sánh Decision Tree



In [ ]:
NGUYEN = Path('../decision_tree/14-11-2026-Nguyen-decision-tree-clean/data/valid_predictions.csv')
ng = pd.read_csv(NGUYEN)


assert len(ng) == len(y_valid)
assert np.allclose(ng['SalePrice'].values, y_valid.values)

pred_mine = np.expm1(models['Decision Tree'].predict(X_valid))

compare = pd.DataFrame({
    'Baseline (processed, log1p, mặc định)': evaluate(y_valid, pred_mine),
    'Nguyên Run 1 (feature sạch, mặc định)': evaluate(y_valid, ng['pred_run1']),
    'Nguyên Run 3 (feature sạch, max_depth=6)': evaluate(y_valid, ng['pred_run3']),
}).T
compare = compare[['RMSLE', 'RMSE', 'MAE', 'R2']].round({'RMSLE': 4, 'RMSE': 0, 'MAE': 0, 'R2': 4})
compare['Kaggle public score'] = [0.2247, None, None]  # Nguyên chưa nộp Kaggle (theo nhật ký 14/11)
compare.to_csv('decision_tree_comparison.csv')
compare

,RMSLE,RMSE,MAE,R2,Kaggle public score
"Baseline (processed, log1p, mặc định)",0.2269,43622.0,29929.0,0.6555,0.2247
"Nguyên Run 1 (feature sạch, mặc định)",0.2125,40259.0,27923.0,0.7066,NaN
"Nguyên Run 3 (feature sạch, max_depth=6)",0.2081,38299.0,26930.0,0.7345,NaN


## 7. Nhận xét so sánh Decision Tree

- Hai bản chấm trên cùng một tập valid (292 dòng, đã kiểm tra `SalePrice` trùng khớp từng dòng), nên so sánh điểm valid là công bằng.
- Bản của Nguyên tốt hơn baseline: Run 1 (mặc định) giảm RMSLE từ 0.2272 xuống 0.2125 và RMSE từ 43,635 xuống 40,259 USD; Run 3 (`max_depth=6`) còn 0.2081 và 38,299 USD (R² 0.7345 so với 0.6553).
- Khác biệt gộp nhiều thay đổi cùng lúc: bộ feature (processed 146 cột so với feature sạch 80 cột mã hoá thành 288 cột), cách xử lý target (`log1p` so với USD gốc) và độ sâu cây (Run 3). Chỉ với một lần chia dữ liệu và valid 292 dòng, chưa thể kết luận yếu tố nào tạo ra chênh lệch.
- Run 3 giới hạn độ sâu nên bớt overfitting rõ: RMSE train 24,568 USD, trong khi cây mặc định của baseline gần như học thuộc tập train (232 USD).
- Điểm Kaggle: chỉ baseline có (0.2247). Theo nhật ký của Nguyên (14/11), bản của bạn ấy chưa nộp Kaggle, nên chưa so được ở tập test thật.